# LSTM Gesture Classifier

This notebook trains a **stacked LSTM** model for dynamic gesture recognition using time-series data from a sensory data glove.

**Data format:**  
Each gesture trial is a CSV file (~5 s, ~22 Hz) stored in a folder whose name is the gesture label.  
Column naming convention: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`

**Pipeline overview:**
1. Select sensor channels via config flags
2. Load + filter + resample + normalise all trials
3. Stratified train / val / test split
4. Build and train a stacked LSTM with early stopping
5. Evaluate with confusion matrix and per-class classification report
6. Save model weights and a JSON results summary

---
**Column naming convention:**
```
{hand}_{segment}_{loc}_{channel}

Examples:
  left_thumb_mid_ax          right_index_prox_pitch
  left_thumb_flex_mcp        right_palm_az
  left_wrist_roll
```
Quaternion columns (`qw`, `qx`, `qy`, `qz`) are **always excluded**.

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'scipy',
    'matplotlib',
    'seaborn',
]

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
)
print('All dependencies installed.')

---
## Cell 2 — Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import json
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Numerical / data ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Project utilities ─────────────────────────────────────────────────────────
# Ensure the ML/ directory is on the path so 'utils' is importable
ML_DIR = Path(os.path.abspath(''))
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from utils.data_loader import build_dataset, split_dataset, build_column_groups

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'Working directory  : {ML_DIR}')

---
## Cell 3 — CONFIG

> **Edit all experiment settings here.** Every tunable parameter lives in this cell — no other cell needs to be modified for standard experiments.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                          EXPERIMENT CONFIGURATION                          ║
# ║  Edit settings in this cell to control every aspect of the LSTM pipeline.  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'       # Path to the dataset root folder.
                            # Each sub-folder name becomes the gesture class label.
FS_HZ     = 22.0            # Sampling rate of the glove in Hz.
                            # Used for filter design and documentation only.

# ── COLUMN SELECTION ──────────────────────────────────────────────────────────
# Set True / False to include / exclude each sensor group.
# This controls exactly which channels are fed into the model.

USE_LEFT_HAND  = True       # Include all channels from the LEFT glove.
USE_RIGHT_HAND = True       # Include all channels from the RIGHT glove.

# Per-finger IMU locations (applies to both hands when the hand flag is True)
USE_DISTAL_IMU    = True    # 'mid' sensor location — distal phalanx IMU
                            # Channels: ax, ay, az, pitch, roll, yaw
USE_PROXIMAL_IMU  = True    # 'prox' sensor location — proximal phalanx IMU
                            # Channels: ax, ay, az, pitch, roll, yaw

# IMU channel sub-types (applies to finger, palm, and wrist IMUs)
USE_ACCELEROMETER = True    # Linear acceleration channels: ax, ay, az
USE_ORIENTATION   = True    # Orientation channels: pitch, roll, yaw
                            # NOTE: at least one of the two must remain True

# Flex sensors — one pair of bend sensors per finger
USE_FLEX_SENSORS  = True    # MCP (metacarpo-phalangeal) and PIP (proximal
                            # inter-phalangeal) flex sensor readings.

# Palm and Wrist IMUs
USE_PALM_IMU  = True        # Back-of-palm IMU: ax, ay, az, pitch, roll, yaw
USE_WRIST_IMU = True        # Wrist / forearm IMU: ax, ay, az, pitch, roll, yaw

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied before resampling (on raw channel values).
#
# Options:
#   'none'           – no filtering (fastest)
#   'butterworth_lp' – 4th-order Butterworth low-pass  (cutoff 6 Hz)
#   'butterworth_hp' – 4th-order Butterworth high-pass (cutoff 0.5 Hz)
#   'butterworth_bp' – 4th-order Butterworth band-pass (0.5 – 6 Hz)
#   'moving_average' – causal moving average (window 5)
#   'savgol'         – Savitzky-Golay (window 11, polynomial order 3)
#   'median'         – median filter (kernel 5)
FILTER_TYPE = 'butterworth_lp'

# Resampling target length.
# All trials are resampled to this exact number of time steps via
# linear interpolation before being stacked into the dataset array.
# Default: 5 s × 22 Hz ≈ 110 samples.
TARGET_LEN = 110

# Normalisation method.
#   'minmax' – scales each feature to [0, 1]
#   'zscore' – zero-mean, unit-variance standardisation
#   'none'   – no normalisation (not recommended)
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True      # True  = normalise each trial independently
                            # False = fit a global scaler across all trials

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70          # Fraction of data used for training
VAL_RATIO   = 0.10          # Fraction used for validation (early stopping)
                            # Test fraction = 1 - TRAIN_RATIO - VAL_RATIO = 0.20
RANDOM_SEED = 42            # Fixed seed for reproducible splits

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# Number of units in each stacked LSTM layer.
# Provide a list: one entry = single LSTM, two entries = two stacked LSTMs, etc.
# Example: [128, 64] → LSTM(128) → LSTM(64) → Dense head
LSTM_UNITS        = [128, 64]

DROPOUT_RATE      = 0.3     # Dropout applied after each LSTM layer output
RECURRENT_DROPOUT = 0.2     # Dropout applied to the LSTM recurrent connections
                            # (disabled when CuDNN acceleration is active)

# List of hidden fully-connected layer sizes before the softmax output.
# Example: [64] → one Dense(64, relu) layer then Dense(num_classes, softmax)
DENSE_UNITS = [64]

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 60    # Maximum number of training epochs
BATCH_SIZE          = 32    # Mini-batch size
LEARNING_RATE       = 1e-3  # Initial Adam learning rate
EARLY_STOP_PATIENCE = 10    # Stop training if val_loss does not improve for
                            # this many consecutive epochs

# ── DERIVED (do not edit) ─────────────────────────────────────────────────────
TEST_RATIO = round(1.0 - TRAIN_RATIO - VAL_RATIO, 6)
assert TEST_RATIO > 0, 'TRAIN_RATIO + VAL_RATIO must be strictly less than 1.0'
assert USE_ACCELEROMETER or USE_ORIENTATION, \
    'At least one of USE_ACCELEROMETER / USE_ORIENTATION must be True'

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print('CONFIG loaded successfully.')
print(f'  Data root       : {DATA_ROOT}')
print(f'  Filter          : {FILTER_TYPE}')
print(f'  Target length   : {TARGET_LEN} steps  ({TARGET_LEN / FS_HZ:.1f} s at {FS_HZ} Hz)')
print(f'  Normalisation   : {NORMALIZATION}  (per_sample={PER_SAMPLE_NORM})')
print(f'  Split           : {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} test')
print(f'  LSTM layers     : {LSTM_UNITS}')
print(f'  Dense layers    : {DENSE_UNITS}')
print(f'  Dropout         : {DROPOUT_RATE}  (recurrent {RECURRENT_DROPOUT})')
print(f'  Epochs / LR     : {EPOCHS} / {LEARNING_RATE}')
print(f'  Run timestamp   : {TIMESTAMP}')

---
## Cell 4 — Build Feature Column List from Config

In [ ]:
"""
Translate the boolean CONFIG flags into a concrete list of column names.

build_column_groups() returns a dict  {group_label: [col_name, ...]}
We then filter by the IMU channel sub-type flags (USE_ACCELEROMETER /
USE_ORIENTATION) and flatten the result to a plain list for build_dataset().
"""

# ── 1. Determine which hands / locs to include ────────────────────────────────
hands = []
if USE_LEFT_HAND:  hands.append('left')
if USE_RIGHT_HAND: hands.append('right')

locs = []
if USE_DISTAL_IMU:   locs.append('mid')
if USE_PROXIMAL_IMU: locs.append('prox')

# ── 2. Call the shared utility to build the full group dict ───────────────────
col_groups = build_column_groups(
    hands        = hands,
    locs         = locs,
    include_flex  = USE_FLEX_SENSORS,
    include_palm  = USE_PALM_IMU,
    include_wrist = USE_WRIST_IMU,
    include_imu   = True,          # always True; sub-type filtering done below
)

# ── 3. Filter IMU channels by sub-type ───────────────────────────────────────
ACCEL_CHANNELS       = {'ax', 'ay', 'az'}
ORIENTATION_CHANNELS = {'pitch', 'roll', 'yaw'}

def keep_channel(col_name: str) -> bool:
    """Return True if this column's channel type matches the config flags."""
    # Flex sensors are not IMU channels — always keep when enabled.
    if 'flex_' in col_name:
        return True
    # For IMU columns, check the trailing channel token.
    suffix = col_name.rsplit('_', 1)[-1]
    if suffix in ACCEL_CHANNELS and not USE_ACCELEROMETER:
        return False
    if suffix in ORIENTATION_CHANNELS and not USE_ORIENTATION:
        return False
    return True

# Apply filter to each group and rebuild
filtered_groups = {}
for group_name, cols in col_groups.items():
    kept = [c for c in cols if keep_channel(c)]
    if kept:
        filtered_groups[group_name] = kept

# ── 4. Flatten to a single ordered list ──────────────────────────────────────
FEATURE_COLS = [col for cols in filtered_groups.values() for col in cols]

# ── 5. Print a grouped summary ────────────────────────────────────────────────
print(f'Total feature columns requested: {len(FEATURE_COLS)}\n')
print(f'{"Group":<35}  {"Count":>5}  Columns')
print('-' * 100)
for group_name, cols in filtered_groups.items():
    col_preview = ', '.join(cols[:4]) + (' ...' if len(cols) > 4 else '')
    print(f'{group_name:<35}  {len(cols):>5}  {col_preview}')

print()
print('Active sensor groups summary:')
print(f'  Hands           : {hands}')
print(f'  Finger IMU locs : {locs}')
print(f'  Accelerometer   : {USE_ACCELEROMETER}')
print(f'  Orientation     : {USE_ORIENTATION}')
print(f'  Flex sensors    : {USE_FLEX_SENSORS}')
print(f'  Palm IMU        : {USE_PALM_IMU}')
print(f'  Wrist IMU       : {USE_WRIST_IMU}')

---
## Cell 5 — Load and Preprocess Dataset

In [ ]:
"""
Load every CSV trial under DATA_ROOT, apply the temporal filter, resample
to TARGET_LEN time steps, and normalise.

Returns:
    X              : float32 array of shape (N, TARGET_LEN, C)
    y              : int64 array of shape  (N,)
    CLASS_NAMES    : list of gesture label strings (index = class integer)
    feature_cols   : list of column names actually present in the data
"""

X, y, CLASS_NAMES, feature_cols = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = FEATURE_COLS,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
    exclude_quat    = True,
)

NUM_CLASSES   = len(CLASS_NAMES)
NUM_FEATURES  = X.shape[2]

print(f'\nDataset summary:')
print(f'  X shape        : {X.shape}  (samples × timesteps × features)')
print(f'  y shape        : {y.shape}')
print(f'  Classes ({NUM_CLASSES})    : {CLASS_NAMES}')
print(f'  Features used  : {NUM_FEATURES} (of {len(FEATURE_COLS)} requested)')

# Warn if some requested columns were absent in the data
missing = [c for c in FEATURE_COLS if c not in feature_cols]
if missing:
    print(f'  WARNING: {len(missing)} requested columns not found in data and were skipped:')
    for m in missing[:10]:
        print(f'    {m}')
    if len(missing) > 10:
        print(f'    ... and {len(missing) - 10} more')

# ── Class distribution bar chart ──────────────────────────────────────────────
unique, counts = np.unique(y, return_counts=True)
label_names = [CLASS_NAMES[i] for i in unique]

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES), 4))
bars = ax.bar(label_names, counts, color='#20808D', edgecolor='white', linewidth=0.8)
ax.set_xlabel('Gesture Class', fontsize=12)
ax.set_ylabel('Number of Trials', fontsize=12)
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Annotate bar heights
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved to results/class_distribution.png')

---
## Cell 6 — Train / Val / Test Split

In [ ]:
"""
Stratified split into train / val / test subsets.
Labels are one-hot encoded for Keras categorical crossentropy.
"""

os.makedirs('results', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

(X_train, y_train_int), (X_val, y_val_int), (X_test, y_test_int) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

# One-hot encode integer labels → shape (N, NUM_CLASSES)
y_train = to_categorical(y_train_int, num_classes=NUM_CLASSES)
y_val   = to_categorical(y_val_int,   num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_int,  num_classes=NUM_CLASSES)

print('Split shapes:')
print(f'  X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}     y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}    y_test  : {y_test.shape}')

# Verify class balance in each split
for split_name, split_y in [('Train', y_train_int), ('Val', y_val_int), ('Test', y_test_int)]:
    u, c = np.unique(split_y, return_counts=True)
    balance = ', '.join(f'{CLASS_NAMES[i]}:{n}' for i, n in zip(u, c))
    print(f'  {split_name:6s} class counts: {balance}')

---
## Cell 7 — Model Definition

In [ ]:
"""
Build a stacked LSTM model.

Architecture
------------
Input  →  [LSTM(units, return_sequences=True)  →  Dropout]  × (n-1 layers)
       →   LSTM(units, return_sequences=False) →  Dropout
       →  [Dense(units, activation='relu')     →  Dropout]  × n_dense
       →   Dense(NUM_CLASSES, activation='softmax')

Notes
-----
- return_sequences=True is required for all LSTM layers except the last
  so that the next LSTM layer receives a full time-step sequence.
- recurrent_dropout > 0 disables CuDNN kernel; use 0.0 for GPU speed.
"""

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def build_lstm_model(
    input_shape,
    num_classes,
    lstm_units,
    dropout_rate,
    recurrent_dropout,
    dense_units,
):
    """
    Construct a configurable stacked LSTM classifier.

    Parameters
    ----------
    input_shape       : (timesteps, features)
    num_classes       : number of gesture classes (output units)
    lstm_units        : list[int] — units per LSTM layer
    dropout_rate      : float in [0, 1) — output dropout after each LSTM
    recurrent_dropout : float in [0, 1) — recurrent dropout inside LSTM
    dense_units       : list[int] — units per hidden FC layer

    Returns
    -------
    keras.Model
    """
    inputs = keras.Input(shape=input_shape, name='glove_input')
    x = inputs

    # ── Stacked LSTM layers ────────────────────────────────────────────────────
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)  # True for all but the last layer
        x = layers.LSTM(
            units,
            return_sequences  = return_seq,
            dropout           = dropout_rate,
            recurrent_dropout = recurrent_dropout,
            name              = f'lstm_{i+1}',
        )(x)
        x = layers.Dropout(dropout_rate, name=f'dropout_lstm_{i+1}')(x)

    # ── Fully-connected head ───────────────────────────────────────────────────
    for j, units in enumerate(dense_units):
        x = layers.Dense(units, activation='relu', name=f'dense_{j+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'dropout_dense_{j+1}')(x)

    # ── Output layer ──────────────────────────────────────────────────────────
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='LSTM_Gesture_Classifier')


model = build_lstm_model(
    input_shape       = (TARGET_LEN, NUM_FEATURES),
    num_classes       = NUM_CLASSES,
    lstm_units        = LSTM_UNITS,
    dropout_rate      = DROPOUT_RATE,
    recurrent_dropout = RECURRENT_DROPOUT,
    dense_units       = DENSE_UNITS,
)

model.summary()

---
## Cell 8 — Training

In [ ]:
"""
Compile and train the LSTM model.

Callbacks
---------
EarlyStopping     — restores best weights; stops after EARLY_STOP_PATIENCE
                    epochs of no improvement in val_loss.
ReduceLROnPlateau — halves the learning rate when val_loss plateaus for 5
                    epochs; minimum LR = 1e-6.
"""

# ── Compile ───────────────────────────────────────────────────────────────────
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 5,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
]

# ── Train ─────────────────────────────────────────────────────────────────────
print(f'Training on {len(X_train)} samples  |  '
      f'Validating on {len(X_val)} samples  |  '
      f'Max epochs: {EPOCHS}')

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

# ── Learning curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epochs_ran, history.history['loss'],     label='Train loss',
             color='#20808D', linewidth=2)
axes[0].plot(epochs_ran, history.history['val_loss'], label='Val loss',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history.history['accuracy'],     label='Train acc',
             color='#20808D', linewidth=2)
axes[1].plot(epochs_ran, history.history['val_accuracy'], label='Val acc',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History — LSTM Gesture Classifier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'results/training_curves_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Training complete. Best val_loss: '
      f'{min(history.history["val_loss"]):.4f}  |  '
      f'Best val_acc: {max(history.history["val_accuracy"]):.4f}')

---
## Cell 9 — Evaluation

In [ ]:
"""
Evaluate the trained model on the held-out test set.

Outputs
-------
- Overall test accuracy and loss
- Per-class precision, recall, F1 (classification_report)
- Confusion matrix heatmap
"""

# ── Test loss and accuracy ─────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f'Test accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print(f'Test loss     : {test_loss:.4f}')

# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_proba = model.predict(X_test, verbose=0)          # (N, NUM_CLASSES)
y_pred_int   = np.argmax(y_pred_proba, axis=1)           # predicted class indices

# ── Classification report ─────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('Classification Report')
print('=' * 60)
report_str = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES, zero_division=0
)
print(report_str)

# Store dict version for JSON saving later
report_dict = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test_int, y_pred_int)

# Normalise to percentages for readability
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES + 2), max(6, NUM_CLASSES)))

sns.heatmap(
    cm_norm,
    annot      = True,
    fmt        = '.1f',
    cmap       = 'Blues',
    xticklabels = CLASS_NAMES,
    yticklabels = CLASS_NAMES,
    linewidths  = 0.5,
    cbar_kws   = {'label': 'Recall (%)'},
    ax         = ax,
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix — Test Accuracy {test_acc * 100:.2f}%',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'results/confusion_matrix_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Confusion matrix saved to results/confusion_matrix_{TIMESTAMP}.png')

---
## Cell 10 — Save Model and Results

In [ ]:
"""
Persist the trained model and a comprehensive results summary.

Saved artefacts
---------------
saved_models/lstm_{TIMESTAMP}.keras   — full Keras model (weights + architecture)
results/lstm_{TIMESTAMP}.json         — config, metrics, classification report,
                                        training history summary
"""

MODEL_PATH   = f'saved_models/lstm_{TIMESTAMP}.keras'
RESULTS_PATH = f'results/lstm_{TIMESTAMP}.json'

# ── Save Keras model ──────────────────────────────────────────────────────────
model.save(MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')

# ── Assemble results dict ─────────────────────────────────────────────────────
results = {
    'timestamp'    : TIMESTAMP,
    'model_path'   : MODEL_PATH,

    # Exact config snapshot
    'config': {
        'DATA_ROOT'          : DATA_ROOT,
        'FS_HZ'              : FS_HZ,
        'USE_LEFT_HAND'      : USE_LEFT_HAND,
        'USE_RIGHT_HAND'     : USE_RIGHT_HAND,
        'USE_DISTAL_IMU'     : USE_DISTAL_IMU,
        'USE_PROXIMAL_IMU'   : USE_PROXIMAL_IMU,
        'USE_ACCELEROMETER'  : USE_ACCELEROMETER,
        'USE_ORIENTATION'    : USE_ORIENTATION,
        'USE_FLEX_SENSORS'   : USE_FLEX_SENSORS,
        'USE_PALM_IMU'       : USE_PALM_IMU,
        'USE_WRIST_IMU'      : USE_WRIST_IMU,
        'FILTER_TYPE'        : FILTER_TYPE,
        'TARGET_LEN'         : TARGET_LEN,
        'NORMALIZATION'      : NORMALIZATION,
        'PER_SAMPLE_NORM'    : PER_SAMPLE_NORM,
        'TRAIN_RATIO'        : TRAIN_RATIO,
        'VAL_RATIO'          : VAL_RATIO,
        'RANDOM_SEED'        : RANDOM_SEED,
        'LSTM_UNITS'         : LSTM_UNITS,
        'DROPOUT_RATE'       : DROPOUT_RATE,
        'RECURRENT_DROPOUT'  : RECURRENT_DROPOUT,
        'DENSE_UNITS'        : DENSE_UNITS,
        'EPOCHS'             : EPOCHS,
        'BATCH_SIZE'         : BATCH_SIZE,
        'LEARNING_RATE'      : LEARNING_RATE,
        'EARLY_STOP_PATIENCE': EARLY_STOP_PATIENCE,
    },

    # Dataset info
    'dataset': {
        'num_samples'       : int(X.shape[0]),
        'num_timesteps'     : int(X.shape[1]),
        'num_features'      : int(X.shape[2]),
        'num_classes'       : NUM_CLASSES,
        'class_names'       : CLASS_NAMES,
        'feature_cols_used' : feature_cols,
        'n_train'           : int(len(X_train)),
        'n_val'             : int(len(X_val)),
        'n_test'            : int(len(X_test)),
    },

    # Evaluation metrics
    'test_accuracy'       : float(test_acc),
    'test_loss'           : float(test_loss),
    'classification_report': report_dict,

    # Training history summary
    'training_history': {
        'epochs_trained'    : len(history.history['loss']),
        'final_train_loss'  : float(history.history['loss'][-1]),
        'final_val_loss'    : float(history.history['val_loss'][-1]),
        'best_val_loss'     : float(min(history.history['val_loss'])),
        'final_train_acc'   : float(history.history['accuracy'][-1]),
        'final_val_acc'     : float(history.history['val_accuracy'][-1]),
        'best_val_acc'      : float(max(history.history['val_accuracy'])),
    },
}

# ── Write JSON ────────────────────────────────────────────────────────────────
with open(RESULTS_PATH, 'w') as fh:
    json.dump(results, fh, indent=2)

print(f'Results saved to: {RESULTS_PATH}')
print(f'\nSummary:')
print(f'  Test accuracy       : {test_acc * 100:.2f}%')
print(f'  Test loss           : {test_loss:.4f}')
print(f'  Epochs trained      : {results["training_history"]["epochs_trained"]}')
print(f'  Best val accuracy   : {results["training_history"]["best_val_acc"] * 100:.2f}%')

---
## Cell 11 — Inference Example *(optional)*

Load a single CSV trial, run it through the full preprocessing pipeline, and print the predicted gesture label.  
Set `INFERENCE_CSV` to the path of any valid glove CSV file.

In [ ]:
"""
Single-sample inference demo.

Steps
-----
1. Load the CSV with load_csv(), selecting only the feature columns used
   during training.
2. Apply the same temporal filter as training.
3. Resample to TARGET_LEN time steps.
4. Normalise with the same method as training.
5. Add a batch dimension and call model.predict().
6. Print the predicted class label and confidence scores.
"""

from utils.data_loader import (
    load_csv,
    preprocess_dataframe,
    resample_to_length,
    normalize,
)

# ── Set the path to the CSV you want to run inference on ──────────────────────
INFERENCE_CSV = '../data/example_gesture/trial_001.csv'  # <-- edit this path

# ── Check if file exists before proceeding ────────────────────────────────────
if not Path(INFERENCE_CSV).exists():
    print(f'[SKIP] Inference CSV not found: {INFERENCE_CSV}')
    print('Set INFERENCE_CSV to a valid path to run this cell.')
else:
    # 1. Load and select columns
    df_infer = load_csv(INFERENCE_CSV, feature_cols=feature_cols, exclude_quat=True)

    # Warn about any missing columns
    loaded_cols = df_infer.columns.tolist()
    missing_infer = [c for c in feature_cols if c not in loaded_cols]
    if missing_infer:
        print(f'WARNING: {len(missing_infer)} expected columns absent in inference CSV.')

    # 2. Temporal filter
    if FILTER_TYPE != 'none':
        df_infer = preprocess_dataframe(df_infer, filter_type=FILTER_TYPE, fs=FS_HZ)

    # 3. Resample
    arr_infer = df_infer.values.astype(np.float32)
    arr_infer = resample_to_length(arr_infer, TARGET_LEN)       # (TARGET_LEN, C)

    # 4. Normalise (per-sample: add and immediately remove the batch dim)
    arr_infer = arr_infer[np.newaxis, ...]                      # (1, T, C)
    if NORMALIZATION != 'none':
        arr_infer = normalize(arr_infer, method=NORMALIZATION, per_sample=True)

    # 5. Predict
    proba = model.predict(arr_infer, verbose=0)[0]              # (NUM_CLASSES,)
    pred_idx   = np.argmax(proba)
    pred_label = CLASS_NAMES[pred_idx]
    confidence = proba[pred_idx] * 100

    # 6. Report
    print(f'Input file   : {INFERENCE_CSV}')
    print(f'Predicted    : {pred_label}  (confidence {confidence:.1f}%)')
    print('\nPer-class probabilities:')
    for cls, p in sorted(zip(CLASS_NAMES, proba), key=lambda t: -t[1]):
        bar = '█' * int(p * 30)
        print(f'  {cls:<30} {p * 100:5.1f}%  {bar}')